# Lab 3 — Structured tickets and the tool loop

*Day 2, hours 2 and 4 · 2 × 50 minutes · pairs*

::: {.callout-note appearance="simple"}
**Objective** — Part A: extract validated `ServiceTicket` objects from messy
bilingual citizen messages and measure the schema-pass rate. Part B: wire three
tools and the bounded loop, and pass the negative-test suite.

**Before you start** — Module 2's lab complete. `data/citizen_messages_50.jsonl` —
bilingual and deliberately messy: dialect, missing hamzas, Arabic-Indic digits,
mixed script, one 400-word polite ramble.

**You finish with** — six numbers in `BENCHMARKS.md`, an invented-field audit at
zero, and the tool-safety suite green.
:::

In [1]:
import os, pathlib, sys, re, subprocess, urllib.request, json

# pytest and ruff colour their output; those escapes render as noise once the
# notebook is published, so they come off here rather than per command.
ANSI = re.compile(chr(27) + r"\[[0-9;]*m")

for cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (cand / "src" / "murshid").is_dir():
        os.chdir(cand); break
    if (cand / "murshid" / "src" / "murshid").is_dir():
        os.chdir(cand / "murshid"); break

sys.path.insert(0, "src")
os.environ["PYTHONUTF8"] = "1"
os.environ.setdefault("PYTHONPATH", "src")

def run(*args, quiet_logs=True):
    """Run a course command and print what it printed.

    quiet_logs drops the structured log lines so the boxed summary is readable;
    pass quiet_logs=False when the log IS the lesson.
    """
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    text = ANSI.sub("", out.stdout + out.stderr)
    if quiet_logs:
        # Structured logs come in two shapes — the console format on a laptop and
        # JSON lines in the container — so drop both, rather than whichever one
        # the machine that built this notebook happened to emit.
        def _is_log(line):
            if line.startswith("20") and "[" in line[:40]:
                return True
            return line.lstrip().startswith('{"') and (
                '"stage"' in line or '"event"' in line or '"logger"' in line)
        text = "\n".join(l for l in text.splitlines() if not _is_log(l))
    else:
        # The log is the lesson here, but not all of it: assistant_built and the
        # per-call llm_cost records are plumbing, and they are also the widest
        # lines on the page. Keep the retries, the failover and the refusals.
        NOISE = ("llm_cost", "assistant_built")
        text = "\n".join(l for l in text.splitlines()
                          if not any(n in l for n in NOISE))
    print(text.strip())
    return out.returncode

# The gateway is 127.0.0.1 on a laptop and `gateway` inside compose, so take it
# from the same environment variable the application routes through rather than
# hardcoding a host that is only right in one of the two places.
GATEWAY = os.environ.get("MURSHID_PRIMARY_BASE_URL", "http://127.0.0.1:8080/v1")
GATEWAY = GATEWAY.rsplit("/v1", 1)[0].rstrip("/")

def fault(payload):
    """Fault injection on the course gateway: the 429 storm and the outage drill."""
    req = urllib.request.Request(
        GATEWAY + "/admin/fault", method="POST",
        data=json.dumps(payload).encode(), headers={"content-type": "application/json"})
    with urllib.request.urlopen(req, timeout=5) as r:
        return json.load(r)

def gateway_stats():
    with urllib.request.urlopen(GATEWAY + "/admin/stats", timeout=5) as r:
        return json.load(r)

try:
    with urllib.request.urlopen(GATEWAY + "/healthz", timeout=3) as r:
        print("gateway:", json.load(r)["models"])
except Exception:
    print(f"gateway at {GATEWAY} is NOT answering — start it first:")
    print("   make gateway      (or)   docker compose up -d gateway")
print("cwd:", pathlib.Path.cwd())

gateway: ['course-flagship', 'course-small', 'course-anthropic', 'murshid-onprem']
cwd: /srv


# Part A — the ticket

## 1 · The contract (10 min)

The validators carry the rules no JSON Schema can express. Read both.

In [2]:
import inspect
from murshid.domain.ticket import Applicant
src = inspect.getsource(Applicant)
print(src[src.index("@field_validator"):][:1100])

@field_validator("national_id")
    @classmethod
    def valid_national_id(cls, v: str | None) -> str | None:
        if v is not None and not (len(v) == 10 and v.isdigit() and v[0] in "12"):
            raise ValueError("must be 10 digits starting with 1 (citizen) or 2 (resident)")
        return v

    @field_validator("phone")
    @classmethod
    def valid_phone(cls, v: str | None) -> str | None:
        if v is None:
            return v
        digits = v.replace(" ", "").replace("-", "")
        if not (digits.startswith(("+9665", "05", "9665")) and sum(c.isdigit() for c in digits) >= 9):
            raise ValueError("must be a Saudi mobile number, e.g. +9665XXXXXXXX")
        return digits



`if v is None: return v` — absent is legal, *invented* is not, and that distinction
is the never-invent rule expressed in a type. The phone validator normalises before
checking, so `05x xxx xxxx` and `+9665xxxxxxxx` are the same number to this
contract, and its error message names the expected shape because that message is
what the repair turn sees.

Confirm the contract still fits the strict-mode subset:

In [3]:
run("scripts/schema_check.py")

────────────────────────────────────────────────────────────────────────
schema-check | strict-mode subset
────────────────────────────────────────────────────────────────────────
  OK  service_ticket
  OK  applicant
  OK  booking_request
  OK  guard_verdict
  OK  route_verdict

5/5 contracts strict-safe


0

## 2 · The repair loop (15 min)

Extraction on a rich message, then on one with almost nothing in it.

In [4]:
from murshid.app import build_client
from murshid.config import get_settings
from murshid.pipeline.extract import extract_ticket

client = build_client(get_settings(), "primary")
ticket, outcome = extract_ticket(
    client, "السلام عليكم، اسمي فيصل العتيبي وأبغى أجدد السجل التجاري حقي في الرياض")
print("first try:", outcome.first_try, "| attempts:", outcome.attempts)
print(ticket.model_dump_json(indent=2)[:700])

2026-09-06T12:38:25.893791Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=service_ticket


first try: True | attempts: 1
{
  "service_type": "other",
  "summary_en": "Citizen asks about a government service in Riyadh.",
  "city": "Riyadh",
  "urgency": "routine",
  "language": "ar",
  "applicant": {
    "full_name": "فيصل العتيبي وأبغى أجدد",
    "national_id": null,
    "phone": null
  },
  "needs_human": false
}


In [5]:
ticket2, outcome2 = extract_ticket(client, "كيف أجدد رخصتي التجارية؟")
print("first try:", outcome2.first_try, "| attempts:", outcome2.attempts)
print("national_id:", ticket2.applicant.national_id)
print("phone      :", ticket2.applicant.phone)
print("city       :", ticket2.city)

2026-09-06T12:38:25.991880Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=service_ticket


first try: True | attempts: 1
national_id: None
phone      : None
city       : unknown


`None` is the correct answer. `extract_ticket.v3`'s never-invent rule plus its one
null example are what earn it. **An invented field is a defect; an empty one is a
fact.**

## 3 · Measure the corpus (15 min)

In [6]:
run("scripts/extract_corpus.py", "--audit")

────────────────────────────────────────────────────────────────────────
extract-corpus | route=primary+fallback
────────────────────────────────────────────────────────────────────────
50 messages | first-try pass: 45/50 (90%) | after repair: 48/50 (96%) | escalated: 2
   by language: ar 32/35 (91%) → 34/35 (97%) | en 12/14 (86%) → 13/14 (93%) | mixed 1/1 (100%) → 1/1 (100%)
   invented-field audit: 0 invented across 15 annotated cases
   escalated to human review: 2
     m010: [['urgency']]
     m013: [['urgency']]

   written: eval/out/extract_corpus_default.json


0

All six numbers go in `BENCHMARKS.md`. The two escalations are not a bug: one
repair, then a designed hand-off. **A corpus where nothing ever escalates is not
testing the failure path.**

## 4 · The comparison (10 min)

In [7]:
run("scripts/extract_corpus.py", "--route", "vllm", "--audit")

────────────────────────────────────────────────────────────────────────
extract-corpus | route=vllm
────────────────────────────────────────────────────────────────────────
50 messages | first-try pass: 42/50 (84%) | after repair: 49/50 (98%) | escalated: 1
   by language: ar 28/35 (80%) → 34/35 (97%) | en 13/14 (93%) → 14/14 (100%) | mixed 1/1 (100%) → 1/1 (100%)
   invented-field audit: 0 invented across 15 annotated cases
   escalated to human review: 1
     m031: [['urgency']]

   written: eval/out/extract_corpus_vllm.json


0

::: {.callout-warning}
## Then write a sentence about error bars

Fifty cases carry roughly ±6 points of noise, so a few points between routes after
repair is a coin, not a finding. The differences that *are* real are the first-try
rates and the size of the gap the repair loop closes. Learning which differences
survive their error bars is most of what Module 5 is about.
:::

# Part B — the tool loop

## 5 · Tool descriptions route (10 min)

The smoke suite asserts *which* tools fire, including the cases where none should.

In [8]:
run("scripts/tool_smoke.py")

────────────────────────────────────────────────────────────────────────
tool-smoke
────────────────────────────────────────────────────────────────────────
  OK  status lookup with a reference                   called=['check_application_status'] expected=['check_application_status']
  OK  documents question — must NOT call a tool        called=[] expected=[]
  OK  status question without a reference — must ask, not guess called=[] expected=[]
  OK  booking with everything confirmed                called=['book_appointment'] expected=['book_appointment']
  OK  asks for a human                                 called=['escalate_to_agent'] expected=['escalate_to_agent']

5/5 as expected


0

Now break it on purpose. Descriptions route — one over-broad sentence and the tool
fires on everything.

The smoke script has to run **in this kernel** for the edit to take effect, so
import its `main` rather than shelling out: a subprocess would load its own copy of
the registry and the change would vanish.

In [9]:
import importlib, sys
sys.path.insert(0, "scripts")
tool_smoke = importlib.import_module("tool_smoke")

from murshid.tools import registry
tool = registry.BY_NAME["check_application_status"]
original = tool.description
print("before:", original[:100], "...")

before: Look up the current status of a government application by its reference number (format: two letters  ...


In [10]:
tool.description = "Use for any question about applications."
tool_smoke.main()

2026-09-06T12:38:45.122550Z [info     ] assistant_built                cache=False cascade=False faq_alias=murshid-default faq_prompt=answer_faq.v5 route=primary+fallback routing_enabled=False semantic_cache=False service_alias=murshid-default


2026-09-06T12:38:45.167040Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_verdict trace_id=67f7b323a101


2026-09-06T12:38:45.168479Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_halalas=0.015126 input_tokens=246 intent=guard latency_ms=31.9 model_id=course-small output_tokens=6 prompt_version=input_guard_classifier.v2 route=cheap stage=input_guard trace_id=67f7b323a101


2026-09-06T12:38:45.201254Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=route_verdict


2026-09-06T12:38:45.202300Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_halalas=0.01283 input_tokens=205 intent=router latency_ms=31.1 model_id=course-small output_tokens=6 prompt_version=route_intent.v1 route=cheap stage=router trace_id=


...

2026-09-06T12:38:46.038649Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_halalas=0.01328 input_tokens=205 intent=router latency_ms=31.0 model_id=course-small output_tokens=8 prompt_version=route_intent.v1 route=cheap stage=router trace_id=


2026-09-06T12:38:46.039167Z [info     ] routed                         intent=escalate prompt_version=route_intent.v1


2026-09-06T12:38:46.041710Z [info     ] escalated_to_agent             reason='router sent this conversation to a human'


  BAD booking with everything confirmed                called=['check_application_status'] expected=['book_appointment']
      reply: I couldn't complete that. Ask the citizen to confirm the reference number: two letters and eight dig
  OK  asks for a human                                 called=['escalate_to_agent'] expected=['escalate_to_agent']

3/5 as expected


1

One over-broad sentence and the documents question now fires the status tool.
**Descriptions route** — which is why the `don't` cases in a description are not
padding.

Put it back before moving on; the rest of the lab depends on it.

In [11]:
tool.description = original
tool_smoke.main()

2026-09-06T12:38:46.059073Z [info     ] assistant_built                cache=False cascade=False faq_alias=murshid-default faq_prompt=answer_faq.v5 route=primary+fallback routing_enabled=False semantic_cache=False service_alias=murshid-default


2026-09-06T12:38:46.093676Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=guard_verdict trace_id=ae037d3e31ec


2026-09-06T12:38:46.094721Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_halalas=0.015126 input_tokens=246 intent=guard latency_ms=32.8 model_id=course-small output_tokens=6 prompt_version=input_guard_classifier.v2 route=cheap stage=input_guard trace_id=ae037d3e31ec


2026-09-06T12:38:46.127622Z [info     ] structured_extracted           attempt=1 outcome=first_try schema=route_verdict


2026-09-06T12:38:46.128484Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_halalas=0.01283 input_tokens=205 intent=router latency_ms=31.2 model_id=course-small output_tokens=6 prompt_version=route_intent.v1 route=cheap stage=router trace_id=


...

2026-09-06T12:38:46.884419Z [info     ] llm_cost                       cache_tier= cached_tokens=0 cost_halalas=0.01328 input_tokens=205 intent=router latency_ms=31.4 model_id=course-small output_tokens=8 prompt_version=route_intent.v1 route=cheap stage=router trace_id=


2026-09-06T12:38:46.884866Z [info     ] routed                         intent=escalate prompt_version=route_intent.v1


2026-09-06T12:38:46.886192Z [info     ] escalated_to_agent             reason='router sent this conversation to a human'


  OK  booking with everything confirmed                called=['book_appointment'] expected=['book_appointment']
  OK  asks for a human                                 called=['escalate_to_agent'] expected=['escalate_to_agent']

5/5 as expected


0

## 6 · Bounds and the authorisation gate (15 min)

Every negative test exercises `_execute`, where the *order* of the checks is the
security model.

In [12]:
run("-m", "pytest", "tests/pipeline/test_tool_safety.py", "-v", "--no-header", "-q")

..........                                                               [100%]
10 passed in 0.03s


0

::: {.callout-important}
## Where else could that identity check live?

List the alternatives before reading on: in the prompt; in the tool description; in
the model's good judgement.

Now ask what happens to each under Module 4's injection scenarios. **Every answer
that lives inside the token stream is an answer an attacker can write to.** The
session object is the only one that is not.
:::

Watch the gate refuse a cross-citizen booking directly.

In [13]:
from murshid.domain.session import Session

s = Session(citizen_id="1012345678", identity_verified=True)
for label, args in [("own account ", {"citizen_id": "1012345678"}),
                    ("someone else", {"citizen_id": "2098765432"})]:
    v = s.authorize("book_appointment", args)
    print(f"{label}: allowed={v.allowed} reason={v.reason}")
    if v.user_hint:
        print(f"              hint: {v.user_hint}")

2026-09-06T12:38:48.196834Z [warning  ] authz_cross_citizen_denied     requested_for=2098765432 session=sess_602aca1abf tool=book_appointment


own account : allowed=True reason=
someone else: allowed=False reason=cross_citizen
              hint: I can only act on your own account. Each person books their own appointment from their own account.


The argument is *read*, but `self.citizen_id` is what it is compared against, and
the verdict carries a `user_hint` because a refusal a citizen cannot act on is a
dead end rather than a guardrail.

## 7 · End to end (15 min)

A booking, and then the audit trail it left.

In [14]:
run("-m", "murshid.cli", "ask", "أريد حجز موعد في الأحوال المدنية بالرياض بتاريخ 2026-10-14، أكّد الحجز")

[service → course-flagship via primary] 461ms, 3435 in (3435 cached) / 67 out, 0.765 halalas
تم الحجز. رقم التأكيد APF49DD29B في Riyadh بتاريخ 2026-10-14.


0

## 8 · Commit (10 min)

```bash
git commit -am "feat: validated ticket extraction and the bounded tool loop"
```

## If you finish early

Add parallel execution for read-only calls, and prove side-effecting calls still
serialise. Then argue about the fourth risk class: lab results are read-only
*technically* but sensitive. Does the registry need another class, or does
authorisation already cover it?